In [ ]:
#@markdown Setup dependencies (Colab only)
%%capture
try:
    import papyrus_scripts  # noqa: F401
except ImportError:
    !pip uninstall papyrus-scripts -y
    !pip install --upgrade papyrus-scripts --no-cache-dir
    get_ipython().kernel.do_shutdown(True)

# 🚀 Advanced examples: using Papyrus-scripts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/advanced_querying.ipynb)

This notebook covers two more advanced topics, each demonstrated through both APIs:

1. [🔍 Similarity & substructure search](#1) over the Papyrus compound structures — chemical-space filtering beyond simple column matching.
2. [🧠 QSAR, PCM & DNN modelling](#2) of the bioactivity data — building predictive models directly on curated subsets.

> ⚠️ Both topics need extra data or dependencies: similarity/substructure search needs the compound structures downloaded and the `papyrus-scripts[subsim]` (or `[gpu]`) extra; modelling needs precomputed descriptors and, for XGBoost/DNN respectively, `xgboost`/`papyrus-scripts[dnn]`.

<a id="1"></a>
## 1. 🔍 Similarity & substructure search

Behind the scenes, both APIs below build an indexed [FPSim2](https://github.com/UCLCheminformatics/FPSim2)/RDKit search database (an `FPSubSim2` `.h5` file) from the Papyrus structures:
- **similarity search** ranks compounds by Tanimoto coefficient against a query fingerprint;
- **substructure search** finds compounds containing a query substructure exactly (subgraph isomorphism), not just similar ones.

Building the database requires the structures file to have been downloaded first — `.molecules()` (used below) does that for us.

In [ ]:
from papyrus_scripts import PapyrusDataset

dataset = PapyrusDataset(version='latest', plusplus=True, is3d=False)

# Downloads the compound structures if not already present locally
# _ = dataset.molecules().to_dataframe()

### 🧩 Object-oriented API

`keep_similar_molecules`, `keep_dissimilar_molecules`, `keep_substructure_molecules` and `keep_not_substructure_molecules` transparently build (once) and reuse the underlying `FPSubSim2` search database — no manual setup required.

| Parameter | Meaning |
|---|---|
| `smiles` | one or more query SMILES strings |
| `fp` | fingerprint type for similarity search (default: Morgan/ECFP-like) |
| `threshold` | minimum Tanimoto coefficient to keep (similarity search only) |
| `cuda` | use a GPU-accelerated search engine (default `False`) |

In [ ]:
caffeine = 'Cn1cnc2c1c(=O)n(C)c(=O)n2C'

similar = (dataset
           .keep_similar_molecules(caffeine, threshold=0.6, cuda=False)
           .to_dataframe())
print(f'🔎 Similar to caffeine (Tanimoto ≥ 0.6): {similar.shape[0]} activity points')

substructures = (dataset
                 .keep_substructure_molecules('c1ccccc1OC')  # any molecule containing an anisole moiety
                 .to_dataframe())
print(f'🔎 Containing a benzene ring: {substructures.shape[0]} activity points')

> 🖥️➡️🎮 Pass `cuda=True` to use a GPU-accelerated search engine instead (requires `papyrus-scripts[gpu]` and a CUDA-capable GPU).

### ⚙️ Pre-building the search database explicitly

The filters above call `create_fp_subsim_search()` internally, with default arguments, the first time they run against a dataset whose `.h5` database doesn't exist yet. Call it directly beforehand to control how that database is built:

| Parameter | Meaning |
|---|---|
| `fp` | fingerprint(s) to store (default: Morgan/ECFP-like) |
| `path` | explicit `.h5` output path (default: auto-derived) |
| `progress` | show progress bars while building |
| `njobs` | worker processes for fingerprint computation (`-1` = all cores) |
| `n_shards` | substructure-library shard count (single-process builds only) |
| `pattern_holder_bits` | prescreen size for the substructure library (size vs. query-speed tradeoff) |
| `force` | rebuild even if the database file already exists |

In [ ]:
from papyrus_scripts.fingerprint import MorganFingerprint

# Any of:   FP2Fingerprint, FP3Fingerprint, FP4Fingerprint,
#           AtomPairFingerprint, AvalonFingerprint, MACCSKeysFingerprint,
#           RDKitFingerprint, RDKPatternFingerprint, RDKitTopologicalFingerprint,
#           TopologicalTorsionFingerprint

# Pre-build the database with a custom fingerprint, using 4 CPU cores
db_path = dataset.create_fp_subsim_search(fp=MorganFingerprint(), njobs=4)
# db_path = dataset.create_fp_subsim_search(MorganFingerprint(radius=3, nBits=4096), njobs=8, pattern_holder_bits=128)

print(f'📦 FPSubSim2 database ready at: {db_path}')

### 📚 Function API

The lower-level functions operate on a plain DataFrame/LazyFrame and require an explicit path to an `FPSubSim2` `.h5` database — useful for reusing one database across many independent filter calls, or for combining with the rest of the functional filtering API.

In [ ]:
from papyrus_scripts import consume_chunks, keep_quality, keep_similar, keep_substructure, read_papyrus
from papyrus_scripts.subsim_search import FPSubSim2

# Build the search database explicitly (overwrites it if it already exists)
fpss = FPSubSim2()
# Is created in the current file if the `path` argument is not specified
fpss.create_from_papyrus(version='latest', is3d=False, njobs=4)  # 4 CPU cores
# or load from file
fpss.load('./Papyrus_2024.09.2_FPSubSim2_2D.h5')

sample_data = read_papyrus(is3d=False, plusplus=True, chunksize=1_000_000, source_path=None)
filtered = consume_chunks(keep_quality(sample_data, min_quality='high'), progress=True)

caffeine = 'Cn1cnc2c1c(=O)n(C)c(=O)n2C'

similar = keep_similar(filtered, molecule_smiles=caffeine, fpsubsim2_file=fpss.h5_filename, threshold=0.6)
substructures = keep_substructure(filtered, molecule_smiles='c1ccccc1OC', fpsubsim2_file=fpss.h5_filename)

`FPSubSim2` also gives direct access to the search engines for full control — CPU/GPU/auto-fallback, RAM-light on-disk search, and exact substructure matches with their Papyrus identifiers. To reuse an existing database in a later session, load it first with `fpss.load('/path/to/database.h5')`.

In [ ]:
# cuda=False (default, CPU) | True (GPU, raises if unavailable) | 'auto' (GPU with CPU fallback)
engine = fpss.get_similarity_lib(cuda='auto')
hits = engine.similarity(caffeine, threshold=0.6)
hits.head()

In [ ]:
sub_lib = fpss.get_substructure_lib()
matches = sub_lib.substructure('c1ccccc1OC')
matches.head()

<a id="2"></a>
## 2. 🧠 Modelling: QSAR, PCM and DNN

All modelling functions live under `papyrus_scripts.modelling` and operate on a plain DataFrame (typically produced by either API above) — there is currently no `PapyrusDataset` wrapper for them.

| | QSAR | PCM |
|---|---|---|
| Features | molecular descriptors only | molecular **and** protein descriptors |
| One model per | target | *all* targets in the data at once |
| Best for | a single well-studied target | generalizing (and extrapolating) across related targets |

Let's prepare a small, fast-to-model dataset: activity of the human and rat serotonin transporter.

In [ ]:
from papyrus_scripts import PapyrusDataset

model_dataset = (PapyrusDataset(version='latest', plusplus=True)
                 .keep_accession(['P31645', 'P31652'])  # human & rat SLC6A4
                 .keep_quality('medium')
                 .keep_activity_type(['Ki', 'KD']))

model_data = model_dataset.to_dataframe()
model_data.shape

### 📈 QSAR

QSAR (Quantitative Structure-Activity Relationship) models predict bioactivity from molecular descriptors alone. `qsar()` handles descriptor loading, train/test splitting, cross-validation and evaluation in one call; it defaults to an `xgboost.XGBRegressor`/`XGBClassifier` but accepts any scikit-learn-compatible estimator via `model=`.

A handful of parameters worth understanding before tuning them:

| Parameter | Meaning |
|---|---|
| `num_points` | minimum activity points required for a target to be modelled at all |
| `delta_activity` | minimum spread between the most and least active compound for a target to be modelled |
| `activity_threshold` | active/inactive cutoff used to binarize `endpoint` (classifiers only; ignored by regressors) |
| `split_by` | how the test set is carved out: `'random'`, `'Year'` (temporal), `'cluster'`, `'custom-cluster'`, or `'custom'` |
| `folds` | number of cross-validation folds on the remaining training data |

In [ ]:
import xgboost

from papyrus_scripts.modelling import qsar

reg_results, reg_models = qsar(data=model_data,
                               endpoint='pchembl_value_Mean',
                               num_points=30,
                               delta_activity=2,
                               descriptors='mold2',
                               model=xgboost.XGBRegressor(verbosity=0),
                               folds=5,
                               split_by='Year',
                               split_year=2013,
                               random_state=1234,
                               verbose=True,
                               leave_level=1, # to remove secondary progress bars upon completion
                               )
reg_results

Training a classifier instead is simply a matter of passing a classifier model — `pchembl_value_Mean` is binarized around `activity_threshold` for evaluation.

In [ ]:
cls_results, cls_models = qsar(data=model_data,
                               endpoint='pchembl_value_Mean',
                               descriptors='mold2',
                               activity_threshold=6.5,
                               model=xgboost.XGBClassifier(verbosity=0),
                               folds=5,
                               split_by='Year',
                               split_year=2013,
                               random_state=1234,
                               verbose=True,
                               leave_level=1)
cls_results

### 🧬 PCM

Proteochemometric (PCM) models additionally encode the protein target as descriptors, letting a **single** model generalize across every target present in the data — here, the human and rat transporter jointly, instead of two separate QSAR models.

In [ ]:
from papyrus_scripts.modelling import pcm

pcm_results, pcm_models = pcm(data=model_data,
                              endpoint='pchembl_value_Mean',
                              mol_descriptors='mold2',
                              prot_descriptors='unirep',
                              model=xgboost.XGBRegressor(verbosity=0),
                              folds=5,
                              split_by='Year',
                              split_year=2013,
                              random_state=1234,
                              verbose=True)
pcm_results

### 🎲 y-scrambling

Passing `yscramble=True` to either `qsar()` or `pcm()` randomly permutes the target variable before training. It's a standard sanity check: a model that performs well on real labels should perform close to randomly on scrambled ones — if it doesn't, that's a red flag for data leakage.

In [ ]:
scrambled_results, _ = qsar(data=model_data,
                            endpoint='pchembl_value_Mean',
                            descriptors='mold2',
                            model=xgboost.XGBRegressor(verbosity=0),
                            yscramble=True,
                            random_state=1234,
                            verbose=True,
                            leave_level=1)
scrambled_results

### 🔁 Repeating training over multiple seeds

`qsar()`/`pcm()` do not repeat training internally — `random_state` controls a single train/test split and cross-validation shuffle. To assess how sensitive results are to that choice, call them in a loop over several seeds and inspect the spread:

In [ ]:
import pandas as pd
from tqdm.auto import trange

all_results = []
for seed in trange(5, desc='Fitting 5 replicates'):
    perf, _ = qsar(data=model_data,
                   endpoint='pchembl_value_Mean',
                   descriptors='mold2',
                   model=xgboost.XGBRegressor(verbosity=0),
                   random_state=seed,
                   verbose=True,
                   leave_level=0)
    all_results.append(perf.reset_index().assign(seed=seed))

all_results = pd.concat(all_results, ignore_index=True)
all_results.head()

### 🤖 Deep neural networks (DNN)

`papyrus_scripts.neuralnet` provides single- and multi-task PyTorch/skorch estimators (requires the `papyrus-scripts[dnn]` extra):

| Class | Use case |
|---|---|
| `SingleTaskNNRegressor` / `SingleTaskNNClassifier` | one target, continuous / binary endpoint |
| `MultiTaskNNRegressor` / `MultiTaskNNClassifier` | several targets predicted jointly (`n_task >= 2`) |

> ⚠️ Unlike scikit-learn/XGBoost models, these need `set_architecture()` (and, for early stopping, `set_validation()`) called explicitly before `.fit()`. `qsar()`/`pcm()`'s internal cross-validation loop does **not** call these for you, so DNN models are used standalone below rather than passed as their `model=` argument.

In [ ]:
import polars as pl
from sklearn.model_selection import train_test_split

from papyrus_scripts.neuralnet import SingleTaskNNRegressor

# Reuse the object-oriented API to fetch descriptors for the same filtered compounds
descriptors = model_dataset.molecular_descriptors('mold2').to_dataframe()
# mold2 marks failed computations with an enormous (~1e38) sentinel, not null - unmask before scaling
descriptors = descriptors.with_columns([
    pl.when(pl.col(c).abs() >= 1e30).then(None).otherwise(pl.col(c)).alias(c)
    for c in descriptors.columns if c != 'connectivity'
])
feature_cols = [c for c in descriptors.columns if c != 'connectivity']
data = model_data.join(descriptors, on='connectivity').drop_nulls(subset=feature_cols)

X = data.select(feature_cols).to_numpy()
y = data['pchembl_value_Mean'].to_numpy()

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=1234)

# The model scales the features automatically
reg = SingleTaskNNRegressor(out='./dnn_checkpoints/single_task_regressor', epochs=1500, lr=1e-3,
                             hidden_layers=[512, 128], dropout=0.15)
reg.set_architecture(n_dim=X_train.shape[1])
reg.set_validation(X_valid, y_valid)
reg.fit(X_train, y_train)

predictions = reg.predict(X_valid)

For multi-target models, use `MultiTaskNNRegressor`/`MultiTaskNNClassifier` with `set_architecture(n_dim, n_task)` and a multi-column target array instead — a single model predicts every task at once, sharing the hidden layers.

Reusing the same human/rat serotonin transporter data, pivot it so each compound is one row with a target column per accession (only compounds tested against both are kept):

In [ ]:
from papyrus_scripts.neuralnet import MultiTaskNNRegressor

wide_data = (model_data
             .select(['connectivity', 'target_id', 'pchembl_value_Mean'])
             .pivot(values='pchembl_value_Mean', index='connectivity', on='target_id', aggregate_function='mean')
             .join(descriptors, on='connectivity')
             .drop_nulls())

task_cols = [c for c in wide_data.columns if c not in feature_cols and c != 'connectivity']
X_multi = wide_data.select(feature_cols).to_numpy()
y_multi = wide_data.select(task_cols).to_numpy()
print(f'🧪 {wide_data.shape[0]} compounds tested against all {len(task_cols)} tasks: {task_cols}')

X_train, X_valid, y_train, y_valid = train_test_split(X_multi, y_multi, test_size=0.2, random_state=1234)

multi_reg = MultiTaskNNRegressor(out='./dnn_checkpoints/multi_task_regressor', epochs=1500, lr=1e-3,
                                  hidden_layers=[512, 128], dropout=0.10)
multi_reg.set_architecture(n_dim=X_train.shape[1], n_task=len(task_cols))
multi_reg.set_validation(X_valid, y_valid)
multi_reg.fit(X_train, y_train)

predictions = multi_reg.predict(X_valid)  # shape (n_compounds, n_tasks) — one column per task

### 🔀 Other splitting strategies

`split_by` supports four more strategies besides `'Year'` (used everywhere above) — `pcm()` takes the exact same `split_by`/`cluster_method`/`custom_groups`/`test_set_size` parameters:

| `split_by` | Test set is... |
|---|---|
| `'random'` | a uniformly random `test_set_size` fraction |
| `'cluster'` | built from whichever combination of `cluster_method`-assigned clusters comes closest to `test_set_size` |
| `'custom-cluster'` | the same idea, but using group labels **you** supply via `custom_groups` (e.g. from your own clustering/scaffold-splitting pipeline) |
| `'custom'` | assigned by you directly — `custom_groups` labels each compound `'training'` or `'test'`, no proportion balancing |

`custom_groups` is always a two-column `DataFrame`: the compound id (`connectivity` for the 2D dataset, `InChIKey` for 3D) and either a group label (`'custom-cluster'`) or the literal `'training'`/`'test'` string (`'custom'`).

In [ ]:
from sklearn.cluster import KMeans

random_results, _ = qsar(data=model_data,
                         endpoint='pchembl_value_Mean',
                         descriptors='mold2',
                         model=xgboost.XGBRegressor(verbosity=0),
                         split_by='random',
                         test_set_size=0.3,
                         random_state=1234,
                         verbose=True,
                         leave_level=1)
print(f'🎲 random split — {len(random_results)} fold row(s)')

cluster_results, _ = qsar(data=model_data,
                         endpoint='pchembl_value_Mean',
                         descriptors='mold2',
                         model=xgboost.XGBRegressor(verbosity=0),
                         split_by='cluster',
                         cluster_method=KMeans(n_clusters=10, random_state=1234, n_init=10),
                         test_set_size=0.3,
                         random_state=1234,
                         verbose=True,
                         leave_level=1)
print(f'🧬 cluster split — {len(cluster_results)} fold row(s)')

`'custom-cluster'` and `'custom'` take the group assignment away from `qsar()`/`pcm()` entirely — handy to reuse clusters from an external tool, or a split you designed by hand. Below, `'custom-cluster'` reuses the same kind of KMeans labels as above (just computed by you instead), and `'custom'` reproduces the temporal split from the very first QSAR example, manually, to show the mechanism (approximate: `'custom'` assigns whole compounds, while `'Year'` assigns individual measurements):

In [ ]:
import numpy as np

# 🧩 'custom-cluster': externally-computed group labels, one row per compound
cluster_labels = (KMeans(n_clusters=10, random_state=1234, n_init=10)
                  .fit_predict(descriptors.select(feature_cols).to_numpy()))
custom_clusters = descriptors.select('connectivity').to_pandas()
custom_clusters['cluster'] = cluster_labels

custom_cluster_results, _ = qsar(data=model_data,
                                 endpoint='pchembl_value_Mean',
                                 descriptors='mold2',
                                 model=xgboost.XGBRegressor(verbosity=0),
                                 split_by='custom-cluster',
                                 custom_groups=custom_clusters,
                                 test_set_size=0.3,
                                 random_state=1234,
                                 verbose=True,
                                 leave_level=1)
print(f'🧩 custom-cluster split — {len(custom_cluster_results)} fold row(s)')

# 🗓️ 'custom': explicit training/test labels
years = model_data.to_pandas()[['connectivity', 'Year']].drop_duplicates('connectivity')
custom_labels = years[['connectivity']].copy()
custom_labels['split'] = np.where(pd.to_numeric(years['Year'], errors='coerce') >= 2013, 'test', 'training')

custom_results, _ = qsar(data=model_data,
                         endpoint='pchembl_value_Mean',
                         descriptors='mold2',
                         model=xgboost.XGBRegressor(verbosity=0),
                         split_by='custom',
                         custom_groups=custom_labels,
                         random_state=1234,
                         verbose=True,
                         leave_level=1)
print(f'🗓️ custom split — {len(custom_results)} fold row(s)')

### 🔢 Getting `X_train`/`y_train`/`X_valid`/`y_valid` directly

`qsar()`/`pcm()` run cross-validation internally and return fold results plus fitted models — not the split arrays themselves. To get plain arrays (e.g. to feed the DNN classes above, or to inspect a split), replicate the strategy yourself on the descriptor-merged table. The random-split case is already shown in the 🤖 Deep neural networks section above via `sklearn`'s `train_test_split`; here are the temporal and cluster-based equivalents:

In [ ]:
from papyrus_scripts.modelling import train_test_proportional_group_split

merged = model_data.join(descriptors, on='connectivity').to_pandas()

# 🗓️ Temporal split
years = pd.to_numeric(merged['Year'], errors='coerce')
train_df, valid_df = merged[years < 2013], merged[years >= 2013]
X_train, y_train = train_df[feature_cols].to_numpy(), train_df['pchembl_value_Mean'].to_numpy()
X_valid, y_valid = valid_df[feature_cols].to_numpy(), valid_df['pchembl_value_Mean'].to_numpy()
print(f'🗓️ temporal split — {X_train.shape[0]} train / {X_valid.shape[0]} valid')

# 🧬 Cluster-based split, proportional to test_size
groups = KMeans(n_clusters=10, random_state=1234, n_init=10).fit_predict(merged[feature_cols])
train_df, valid_df, *_ = train_test_proportional_group_split(merged, groups, test_size=0.3, verbose=True)
X_train, y_train = train_df[feature_cols].to_numpy(), train_df['pchembl_value_Mean'].to_numpy()
X_valid, y_valid = valid_df[feature_cols].to_numpy(), valid_df['pchembl_value_Mean'].to_numpy()
print(f'🧬 cluster split — {X_train.shape[0]} train / {X_valid.shape[0]} valid')

---

🎉 That's the full tour! For everyday filtering, see [simple_examples.ipynb](https://github.com/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/simple_examples.ipynb); for matching against the PDB, see [matchRCSB.ipynb](https://github.com/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/matchRCSB.ipynb).